In [ ]:
# 1. サンプルデータセットをダウンロード
%cd /content
!git clone https://github.com/wal-afk/drive_sim
%cd drive_sim
!git pull
!git restore .
!git clean -fd
%cd /content/drive_sim

!pip install -U plotly==6.9

In [ ]:
import yaml

from sim.drive_simulator import (
    CarSim,
)
from sim.vehicle import VehicleProp
from sim.mission_base import MissionBase
from sim.goal import GoalLine, GoalCircle
from sim.drawer import SimDrawer, MissionDrawer
from sim.worlds.type_b_world import type_b_circuit
from sim.sign import Sign

with open("config/type-b.yaml", "r") as f:
    vehicle_config = yaml.safe_load(f)

prop = VehicleProp(**vehicle_config)


In [ ]:
class Tutorial2(MissionBase):
    def __init__(self):
        super().__init__(type_b_circuit, t_max=20)
        self.goals = [
            GoalCircle((0.6, 1.75), 0.1, should_stop=True),
        ]
        self.initial_xy = (0.0, 2.0)
        self.random_d_yaw_deg = 5
        self.set_signs(
            [
                Sign(x=1.2, y=1.5, name="sign1"),
            ]
        )

    @staticmethod
    def command_func(*, move, rotate, search, wait, **kwargs):
        ######## ここから下に「標識の方を向く」プログラムを書こう
        pos = search()
        rotate(w=pos.theta, t=1.0)
        wait()
        ######## ここより上に「標識の方を向く」プログラムを書こう

        ######## ここから下に「標識までの半分の距離だけ前進する」プログラムを書こう
        pos = search()
        move(v=0.2, t=0.5*pos.x/0.2)
        wait()
        ######## ここより上に「標識までの半分の距離だけ前進する」プログラムを書こう
        ######## プログラムを書いた後にセルを実行し結果を確認しよう

sim = CarSim(prop,Tutorial2())
success = sim.run()
if success:
    SimDrawer(sim).show()